## Former code

In [ ]:
import numpy as np
import h5py
from astropy.io import fits
from astropy.table import Table
import pandas as pd

In [ ]:
# This cell converts .fits data file from COSMOS-Web into .csv for later use

master_path= "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:
    hdu.info()
    photom = fits_to_pandas_clean(hdu[1])
    leph   = fits_to_pandas_clean(hdu[2])
    cigale = fits_to_pandas_clean(hdu[4])
    morph  = fits_to_pandas_clean(hdu[5])
    bd     = fits_to_pandas_clean(hdu[6])

cosmos_cat = pd.concat([photom, leph, cigale, morph, bd], axis=1)
output_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_laura.csv"
cosmos_cat.to_csv(output_path, index=False)


In [ ]:
data_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\"
cosmos_cat = pd.read_csv(data_path+"COSMOSWeb_laura.csv") # Data from COSMOS-WEB, converted from .fits to .csv in florah_eval_SFR.ipynb

I the next cell I will select the columns I want, from each extension selected before. This columns will be later saved in a new csv

In [ ]:
# Select the columns I want
columns = ['id', 'radius_sersic', 'ra', 'dec', 'sersic', 'sfr_med', 'mass_med', 'zpdf_med', 'sfr_inst', 'mass', 'morph_flag_f444w', 'b/t_f444w'  ]
cosmos_cat_filtered = cosmos_cat[columns]

# Save new csv
cosmos_cat_filtered.to_csv("C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_Laura_filtered.csv", index=False)


## New code:

In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck15
import astropy.units as u

# 1. Converts .fits data file from COSMOS-Web into .csv for later use
master_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:
    photom = fits_to_pandas_clean(hdu[1])
    leph   = fits_to_pandas_clean(hdu[2])
    cigale = fits_to_pandas_clean(hdu[4])
    morph  = fits_to_pandas_clean(hdu[5])
    bd     = fits_to_pandas_clean(hdu[6])

cosmos_cat = pd.concat([photom, leph, cigale, morph, bd], axis=1)



# 2. Selection and initial cleaning of data
columns_map = {
    'id': 'id',
    'radius_sersic': 'radius_sersic',
    'ra': 'ra',
    'dec': 'dec',
    'sersic': 'sersic',
    'zpdf_med': 'zpdf_med',
    'sfr_inst': 'sfr_raw',
    'mass': 'mass_raw',
    'morph_flag_f444w': 'morphology',
    'b/t_f444w': 'bovert'
}

# Renombramos y filtramos para trabajar solo con lo necesario
cosmos_cat = cosmos_cat[list(columns_map.keys())].rename(columns=columns_map) 

# Convert every value to numeric, transforming errors into NaN
for col in cosmos_cat.columns:
    cosmos_cat[col] = pd.to_numeric(cosmos_cat[col], errors='coerce')



# 3. Logarithmic transformations and physical filters
# Filter impossible values before applying logarithms
cosmos_cat = cosmos_cat[(cosmos_cat['mass_raw'] > 0) & (cosmos_cat['sfr_raw'] > 0)]

cosmos_cat['mass_CIGALE'] = np.log10(cosmos_cat['mass_raw'])
cosmos_cat['sfr_CIGALE'] = np.log10(cosmos_cat['sfr_raw'])



# 4. Pre-calculation - A Phase
# This helps relieve later usage of data, so that astrophysical calculations, such as angular
# diamater distance, that will be once calculated and stored in the output file.
# NOTE: This could compromise RAM usage, but might speed up code
z_array = cosmos_cat['zpdf_med'].values
dist_mpc = Planck15.angular_diameter_distance(z_array).value # Resultado en Mpc


# Calculate physical radius in log10(kpc)
# Formula: Physical_radius = Angular_diameter * Angular_aperture(rad)
cosmos_cat['log_radius_kpc'] = np.log10(dist_mpc * np.deg2rad(cosmos_cat['radius_sersic']) * 1e3)



# 5. Store cleaned file
output_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_Laura_processed.csv"
# Delete rows that contain NaNs in important columns
cosmos_cat.dropna(subset=['mass_CIGALE', 'sfr_CIGALE', 'zpdf_med', 'log_radius_kpc'], inplace=True)

cosmos_cat.to_csv(output_path, index=False)
print(f"Catálogo procesado guardado con {len(cosmos_cat)} filas.")

In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck15
import astropy.units as u

# 1. Converts .fits data file from COSMOS-Web into .csv for later use
master_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:
    photom = fits_to_pandas_clean(hdu[1])
    morph  = fits_to_pandas_clean(hdu[5])

cosmos_cat = pd.concat([photom, morph], axis=1)


# 2. Selection and initial cleaning of data
columns_map = {
    'id': 'id',
    'morph_flag_f444w': 'morphology',
    'delta_f444w': 'delta_f444w'
}

# Renombramos y filtramos para trabajar solo con lo necesario
cosmos_cat = cosmos_cat[list(columns_map.keys())].rename(columns=columns_map) 


obj_id = 737995
# Iterate through the indices of node_features to find the matching ID
for i in range(len(cosmos_cat['id'])):        
    # Safest way to check if ID is in the numpy array
    if cosmos_cat['id'][i] == obj_id:
        index = i
        break
    else: 
        continue


# Convert every value to numeric, transforming errors into NaN
for col in cosmos_cat.columns:
    cosmos_cat[col] = pd.to_numeric(cosmos_cat[col], errors='coerce')

print(cosmos_cat['id'][index])
print(cosmos_cat['morphology'][index])
print(cosmos_cat['delta_f444w'][index])

554341
1
0.5089001247


## Creating another .csv for plotting Cigale's SFHs

In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck15
import astropy.units as u


# 1. Converts .fits data file from COSMOS-Web into .csv for later use
master_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:
    photom = fits_to_pandas_clean(hdu[1])
    cigale = fits_to_pandas_clean(hdu[4])


cosmos_cat = pd.concat([photom, cigale], axis=1)


# 2. Selection and initial cleaning of data
columns_map = {
    'id': 'id',
    'sfh_sfr_bin1': 'sfh_sfr_bin1',
    'sfh_sfr_bin2': 'sfh_sfr_bin2',
    'sfh_sfr_bin3': 'sfh_sfr_bin3',
    'sfh_sfr_bin4': 'sfh_sfr_bin4',
    'sfh_sfr_bin5': 'sfh_sfr_bin5',
    'sfh_sfr_bin6': 'sfh_sfr_bin6',
    'sfh_sfr_bin7': 'sfh_sfr_bin7',
    'sfh_sfr_bin8': 'sfh_sfr_bin8',
    'sfh_sfr_bin9': 'sfh_sfr_bin9',    
    
    'sfh_sfr_bin1_err': 'sfh_sfr_bin1_err',
    'sfh_sfr_bin2_err': 'sfh_sfr_bin2_err',
    'sfh_sfr_bin3_err': 'sfh_sfr_bin3_err',
    'sfh_sfr_bin4_err': 'sfh_sfr_bin4_err',
    'sfh_sfr_bin5_err': 'sfh_sfr_bin5_err',
    'sfh_sfr_bin6_err': 'sfh_sfr_bin6_err',
    'sfh_sfr_bin7_err': 'sfh_sfr_bin7_err',
    'sfh_sfr_bin8_err': 'sfh_sfr_bin8_err',
    'sfh_sfr_bin9_err': 'sfh_sfr_bin9_err',    
    
    'sfh_time_bin1': 'sfh_time_bin1',
    'sfh_time_bin2': 'sfh_time_bin2',
    'sfh_time_bin3': 'sfh_time_bin3',
    'sfh_time_bin4': 'sfh_time_bin4',
    'sfh_time_bin5': 'sfh_time_bin5',
    'sfh_time_bin6': 'sfh_time_bin6',
    'sfh_time_bin7': 'sfh_time_bin7',
    'sfh_time_bin8': 'sfh_time_bin8',
    'sfh_time_bin9': 'sfh_time_bin9',    
    
    'sfh_time_bin1_err': 'sfh_time_bin1_err',
    'sfh_time_bin2_err': 'sfh_time_bin2_err',
    'sfh_time_bin3_err': 'sfh_time_bin3_err',
    'sfh_time_bin4_err': 'sfh_time_bin4_err',
    'sfh_time_bin5_err': 'sfh_time_bin5_err',
    'sfh_time_bin6_err': 'sfh_time_bin6_err',
    'sfh_time_bin7_err': 'sfh_time_bin7_err',
    'sfh_time_bin8_err': 'sfh_time_bin8_err',
    'sfh_time_bin9_err': 'sfh_time_bin9_err',

    'sfh_integrated': 'sfh_integrated',
    'sfh_integrated_err': 'sfh_integrated_err'
}

# 3. Take only the columns we are interested in
cosmos_cat = cosmos_cat[list(columns_map.keys())]


In [ ]:
# 4. Pre-calculation 
# Denormalize sfh and error. Also convert time to the right units
for i in range(1,10):
    # We denormalize sfr firstly
    cosmos_cat['sfh_sfr_bin'+str(i)] = cosmos_cat['sfh_sfr_bin'+str(i)] * cosmos_cat['sfh_integrated']
    cosmos_cat['sfh_sfr_bin'+str(i)] = np.log10(cosmos_cat['sfh_sfr_bin'+str(i)])

    # We calculate true error for sfr by propagating.
    cosmos_cat['sfh_sfr_bin'+str(i)+'_prop_err'] = cosmos_cat['sfh_sfr_bin'+str(i)+'_err'] * cosmos_cat['sfh_integrated'] + cosmos_cat['sfh_sfr_bin'+str(i)] * cosmos_cat['sfh_integrated_err']
    cosmos_cat['sfh_sfr_bin'+str(i)+'_prop_err'] = np.log10(cosmos_cat['sfh_sfr_bin'+str(i)+'_err'])
    # We will also store error in each bin just in case.
    cosmos_cat['sfh_sfr_bin'+str(i)+'_err'] = np.log10(cosmos_cat['sfh_sfr_bin'+str(i)+'_err'] * cosmos_cat['sfh_integrated_err'])
    

    # We clean time data -> From Myr to Gyr
    cosmos_cat['sfh_time_bin'+str(i)] = cosmos_cat['sfh_time_bin'+str(i)] * 10**(-3)
    cosmos_cat['sfh_time_bin'+str(i)+'_err'] = cosmos_cat['sfh_time_bin'+str(i)+'_err'] * 10**(-3)

In [ ]:
# 5. Store cleaned file - One file per galaxy
output_path = r"C:\Users\usuario\Documents\TFG\likelihood_COSMOS_SFR\id_sfh\\"


# We'll process one file per galaxy, containing the data in all of the 9 bins for that galaxy.
for j in range(len(cosmos_cat)):
    all_sfr = []
    all_time = []
    all_sfr_err = []
    all_sfr_prop_err = []
    all_time_err = []

    id_num = str(cosmos_cat['id'][j-1])

    # Find data for j galaxy in all 9 bin files and append value to lists
    for q in range(1,10):
        sfr_in_bin = cosmos_cat['sfh_sfr_bin'+str(q)][j-1]
        all_sfr.append(sfr_in_bin)        
        
        sfr_err_in_bin = cosmos_cat['sfh_sfr_bin'+str(q)+'_err'][j-1]
        all_sfr_err.append(sfr_err_in_bin)        

        sfr_prop_err_in_bin = cosmos_cat['sfh_sfr_bin'+str(q)+'_prop_err'][j-1]
        all_sfr_prop_err.append(sfr_prop_err_in_bin)  
        
        time_in_bin = cosmos_cat['sfh_time_bin'+str(q)][j-1]
        all_time.append(time_in_bin)

        time_err_in_bin = cosmos_cat['sfh_time_bin'+str(q)+'_err'][j-1]
        all_time_err.append(time_err_in_bin) 


    # Combine your lists into a dictionary, then convert to a Pandas DataFrame
    galaxy_data = pd.DataFrame({
        'id': id_num,
        'time': all_time,
        'time_err': all_time_err,
        'sfr': all_sfr,
        'sfr_err': all_sfr_err,
        'sfr_prop_err': all_sfr_prop_err
    })

    # Define the file name
    file_name = f"CIGALE_{id_num}_sfh.csv"

    # Export the DataFrame to a CSV
    galaxy_data.to_csv(output_path+file_name, index=False) # Contains log10(SFR), time in Gyr and corresponding errors

## If we only want to update a specific file, run:

In [16]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck15
import astropy.units as u
from tqdm import tqdm


# 1. Converts .fits data file from COSMOS-Web into .csv for later use
master_path = "C:\\Users\\usuario\\Documents\\TFG\\florah_training_SFR\\COSMOSWeb_mastercatalog_v1.fits"

def fits_to_pandas_clean(hdu):
    tbl = Table(hdu.data)
    names = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
    return tbl[names].to_pandas()

# Selecting only the extensions I will be using
with fits.open(master_path) as hdu:

    photom = fits_to_pandas_clean(hdu[1])
    leph   = fits_to_pandas_clean(hdu[2])
    cigale = fits_to_pandas_clean(hdu[4])



cosmos_cat = pd.concat([photom, leph, cigale], axis=1)


# 2. Selection and initial cleaning of data
columns_map = {
    'id': 'id',

    'zfinal': 'zfinal',
    'zpdf_med': 'zpdf_med',
    'mabs_nuv': 'mabs_nuv',
    'mabs_r': 'mabs_r',
    'mabs_j': 'mabs_j',

    'sfh_sfr_bin1': 'sfh_sfr_bin1',
    'sfh_sfr_bin2': 'sfh_sfr_bin2',
    'sfh_sfr_bin3': 'sfh_sfr_bin3',
    'sfh_sfr_bin4': 'sfh_sfr_bin4',
    'sfh_sfr_bin5': 'sfh_sfr_bin5',
    'sfh_sfr_bin6': 'sfh_sfr_bin6',
    'sfh_sfr_bin7': 'sfh_sfr_bin7',
    'sfh_sfr_bin8': 'sfh_sfr_bin8',
    'sfh_sfr_bin9': 'sfh_sfr_bin9',    
    
    'sfh_sfr_bin1_err': 'sfh_sfr_bin1_err',
    'sfh_sfr_bin2_err': 'sfh_sfr_bin2_err',
    'sfh_sfr_bin3_err': 'sfh_sfr_bin3_err',
    'sfh_sfr_bin4_err': 'sfh_sfr_bin4_err',
    'sfh_sfr_bin5_err': 'sfh_sfr_bin5_err',
    'sfh_sfr_bin6_err': 'sfh_sfr_bin6_err',
    'sfh_sfr_bin7_err': 'sfh_sfr_bin7_err',
    'sfh_sfr_bin8_err': 'sfh_sfr_bin8_err',
    'sfh_sfr_bin9_err': 'sfh_sfr_bin9_err',    
    
    'sfh_time_bin1': 'sfh_time_bin1',
    'sfh_time_bin2': 'sfh_time_bin2',
    'sfh_time_bin3': 'sfh_time_bin3',
    'sfh_time_bin4': 'sfh_time_bin4',
    'sfh_time_bin5': 'sfh_time_bin5',
    'sfh_time_bin6': 'sfh_time_bin6',
    'sfh_time_bin7': 'sfh_time_bin7',
    'sfh_time_bin8': 'sfh_time_bin8',
    'sfh_time_bin9': 'sfh_time_bin9',    
    
    'sfh_time_bin1_err': 'sfh_time_bin1_err',
    'sfh_time_bin2_err': 'sfh_time_bin2_err',
    'sfh_time_bin3_err': 'sfh_time_bin3_err',
    'sfh_time_bin4_err': 'sfh_time_bin4_err',
    'sfh_time_bin5_err': 'sfh_time_bin5_err',
    'sfh_time_bin6_err': 'sfh_time_bin6_err',
    'sfh_time_bin7_err': 'sfh_time_bin7_err',
    'sfh_time_bin8_err': 'sfh_time_bin8_err',
    'sfh_time_bin9_err': 'sfh_time_bin9_err',

    'sfh_integrated': 'sfh_integrated',
    'sfh_integrated_err': 'sfh_integrated_err'
}

# 3. Take only the columns we are interested in
cosmos_cat = cosmos_cat[list(columns_map.keys())]


In [18]:
# 5. Store cleaned file - One file per galaxy
output_path = r"C:\Users\usuario\Documents\TFG\likelihood_COSMOS_SFR\id_sfh_2\\"
id_to_plot = [508710, 525621, 52999, 691283, 233804, 755470, 240207, 666286, 734273, 175080, 734956, 713318, 315646, 402560, 714163, 393092, 587072, 21590, 391150, 144660, 449124, 515418, 518723, 22471, 386299, 585912, 131454, 51727, 676536, 131451, 733398, 361, 588848, 330660, 623991, 328465, 165836, 195598, 294261, 90335, 623707, 105316, 431840, 139731, 310896, 649302, 728441, 131751, 670037, 160309, 446131, 137270, 510377, 135134, 579710, 330386, 130567, 514267, 717051, 118850, 190051, 330499, 500470, 715240, 401378, 400210, 636148]

# We'll process one file per galaxy, containing the data in all of the 9 bins for that galaxy.
for j in tqdm(id_to_plot):
    all_sfr = []
    all_time = []
    all_sfr_err = []
    all_time_err = []
    all_sfh_int = []
    all_sfh_int_err = []
    all_id = []
    zpdf_med =[]
    zfinal = []
    mabs_nuv = []
    mabs_r = []
    mabs_j = []


    # Find data for j galaxy in all 9 bin files and append value to lists
    for q in range(1,10):
        id_num = str(cosmos_cat['id'][j])
        all_id.append(id_num)      

        sfr_in_bin = cosmos_cat['sfh_sfr_bin'+str(q)][j]
        all_sfr.append(sfr_in_bin)        
        
        sfr_err_in_bin = cosmos_cat['sfh_sfr_bin'+str(q)+'_err'][j]
        all_sfr_err.append(sfr_err_in_bin)        
        
        time_in_bin = cosmos_cat['sfh_time_bin'+str(q)][j]
        all_time.append(time_in_bin)

        time_err_in_bin = cosmos_cat['sfh_time_bin'+str(q)+'_err'][j]
        all_time_err.append(time_err_in_bin) 

        sfh_int = cosmos_cat['sfh_integrated'][j]
        all_sfh_int.append(sfh_int) 

        sfh_int_err = cosmos_cat['sfh_integrated_err'][j]
        all_sfh_int_err.append(sfh_int_err)

        zpdf_med_value = cosmos_cat['zpdf_med'][j]
        zpdf_med.append(zpdf_med_value)

        zfinal_value = cosmos_cat['zfinal'][j]
        zfinal.append(zfinal_value)

        mabs_nuv_value = cosmos_cat['mabs_nuv'][j]
        mabs_nuv.append(mabs_nuv_value)

        mabs_r_value = cosmos_cat['mabs_r'][j]
        mabs_r.append(mabs_r_value)

        mabs_j_value = cosmos_cat['mabs_j'][j]
        mabs_j.append(mabs_j_value)




    # Combine your lists into a dictionary, then convert to a Pandas DataFrame
    galaxy_data = pd.DataFrame({
        'id': cosmos_cat['id'][j], # Unique ID of the source
        'time': all_time, # lbt [Myr]
        'time_err': all_time_err, # Error in lbt [Myr]
        'sfr': all_sfr, # Normalized SFR [1/yr]
        'sfr_err': all_sfr_err, # Normalized error in SFR [1/yr]
        'sfh_int': all_sfh_int, # Total integrated star formation history [M_sol]
        'sfh_int_err': all_sfh_int_err, # Error in sfh_integrated [M_sol]
        'zpdf_med': zpdf_med,
        'zfinal': zfinal,
        'mabs_nuv': mabs_nuv,
        'mabs_j': mabs_j,
        'mabs_r': mabs_r
    })

    # Clean NaNs
    galaxy_data = galaxy_data.dropna(axis=0, ignore_index=True)

    # Define the file name
    file_name = f"CIGALE_{id_num}_sfh.csv"

    # Export the DataFrame to a CSV
    galaxy_data.to_csv(output_path+file_name, index=False)


100%|██████████| 67/67 [00:00<00:00, 255.31it/s]
